# Module 23: Distributed Transactions Sagas Outbox — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/saga_orchestrator.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import saga_orchestrator

classes = [n for n, o in inspect.getmembers(saga_orchestrator, inspect.isclass)
           if o.__module__ == 'saga_orchestrator']
functions = [n for n, o in inspect.getmembers(saga_orchestrator, inspect.isfunction)
             if o.__module__ == 'saga_orchestrator']

print('module   : saga_orchestrator')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(saga_orchestrator, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Saga happy path success

This is the module's own `test_saga_happy_path_success` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from saga_orchestrator import (
    InventoryReservationStep,
    OrderCreationStep,
    PaymentProcessingStep,
    SagaOrchestrator,
    SagaStatus,
)

steps = [OrderCreationStep(), PaymentProcessingStep(), InventoryReservationStep()]
orchestrator = SagaOrchestrator(steps)

ctx = {
    "order_id": "ord_1",
    "user_id": "u1",
    "sku": "ITEM_A",
    "amount": 100.0,
    "quantity": 2,
    "accounts_db": {"u1": 500.0},
    "inventory_db": {"ITEM_A": 10},
}

assert orchestrator.run(ctx) is True
assert orchestrator.status == SagaStatus.COMPLETED
assert ctx["accounts_db"]["u1"] == 400.0
assert ctx["inventory_db"]["ITEM_A"] == 8
assert ctx["order_db"]["ord_1"]["status"] == "PENDING"
assert len(orchestrator.logs) == 3

print('PASSED: test_saga_happy_path_success')

## 3. 🔮 Prediction — commit before you run

A saga's third step fails. Predict what the orchestrator must do about steps 1 and 2, and why a database rollback is unavailable to it.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_saga_payment_failure_rollback`, which tests exactly this property.


In [ ]:
steps = [OrderCreationStep(), PaymentProcessingStep(), InventoryReservationStep()]
orchestrator = SagaOrchestrator(steps)

ctx = {
    "order_id": "ord_2",
    "user_id": "u2",
    "sku": "ITEM_A",
    "amount": 1000.0,  # User has only 100
    "quantity": 1,
    "accounts_db": {"u2": 100.0},
    "inventory_db": {"ITEM_A": 10},
}

assert orchestrator.run(ctx) is False
assert orchestrator.status == SagaStatus.COMPENSATED
assert ctx["accounts_db"]["u2"] == 100.0
# Step 1 OrderCreation was executed and must have been compensated
assert ctx["order_db"]["ord_2"]["status"] == "CANCELLED"
# Inventory was never reached
assert ctx["inventory_db"]["ITEM_A"] == 10

print('PASSED: test_saga_payment_failure_rollback')

## 4. Measure it: Saga inventory failure backward compensation

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_saga_inventory_failure_backward_compensation` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

steps = [OrderCreationStep(), PaymentProcessingStep(), InventoryReservationStep()]
orchestrator = SagaOrchestrator(steps)

ctx = {
    "order_id": "ord_3",
    "user_id": "u3",
    "sku": "ITEM_RARE",
    "amount": 50.0,
    "quantity": 5,  # Warehouse only has 2
    "accounts_db": {"u3": 200.0},
    "inventory_db": {"ITEM_RARE": 2},
}

assert orchestrator.run(ctx) is False
assert orchestrator.status == SagaStatus.COMPENSATED
# User had money deducted in Step 2, but compensation restored it back to 200.0
assert ctx["accounts_db"]["u3"] == 200.0
assert ctx["order_db"]["ord_3"]["status"] == "CANCELLED"
assert ctx["inventory_db"]["ITEM_RARE"] == 2

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_saga_inventory_failure_backward_compensation')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(saga_orchestrator) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. A saga has no rollback - it has compensating actions you must write.
2. The outbox pattern makes 'update DB and publish event' a single local transaction.
3. Two-phase commit is unavailable across heterogeneous stores; stop reaching for it.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
